# 📘 Summary: Stage to Enriched Lakehouse Load Process
This notebook performs a structured load from the staging (bronze) layer to the enriched (silver) layer using Slowly Changing Dimension Type 2 (SCD2) logic. The process includes:

1. **Parameter Setup**
- Defines source/target tables, merge keys, and timestamp fields.
2. **Key Generation**
- ELT_UNIQUE_KEY: Hash of merge key columns (used to identify records).
- ELT_SCD_HASH: Hash of merge key columns + timestamp (used to detect changes).
3. **Initial Load**
- If the enriched table doesn’t exist, all records are inserted with Current_Record_Flag = 1.
4. **Incremental Merge**
- If the enriched table exists:
    - Joins new data with current records.
    - Filters for new or updated records.
    - Deduplicates using row_number() to keep the latest version per key.
    - Splits into:
        - Inserts: New keys not seen before.
        - Updates: Existing keys with newer timestamps.
    - Merges into the enriched table:
        - Closes old records (ELT_VALID_TO set, Current_Record_Flag = 0)
        - Inserts new versions (Current_Record_Flag = 1)
5. **Load Timestamp Tracking**
- Captures the latest load timestamp for downstream pipeline use.


## Set Parameters for notebook

In [12]:
p_src_system_abbr = ''
p_src_entity_name = ''
p_merge_key_cols = ''
p_src_modified_field = 'ELT_LOAD_DATETIME_EST'
p_src_db_name = 'Stage_Lakehouse'
p_tgt_db_name = 'Enriched_Lakehouse'
p_tgt_entity_name = ''

StatementMeta(, 5013aa81-5a5c-46a9-ae5c-70a5977c9eff, 14, Finished, Available, Finished)

## Import Necessary Libraries

In [13]:
from pyspark.sql.functions import concat_ws, sha2, col, row_number, lit, when
from pyspark.sql.types import StringType, TimestampType
from pyspark.sql.window import Window
from delta.tables import *
import json

StatementMeta(, 5013aa81-5a5c-46a9-ae5c-70a5977c9eff, 15, Finished, Available, Finished)

## Function to Get Inserts

In [14]:
def get_inserts(tmp_stg_df, silver_df_join, p_src_modified_field):
    inserts_df = tmp_stg_df.join(silver_df_join, tmp_stg_df.ELT_UNIQUE_KEY == silver_df_join.SILVER_DF_ELT_UNIQUE_KEY, "left")
    inserts_df = inserts_df.filter((col("SILVER_DF_ELT_UNIQUE_KEY").isNull()) | 
                                   (col("SILVER_DF_ELT_UNIQUE_KEY").isNotNull() & (col("SILVER_DF_ELT_VALID_FROM") < col(p_src_modified_field))))
    inserts_df = inserts_df.select(tmp_stg_df.columns)
    inserts_df = inserts_df.withColumn("ELT_VALID_FROM", inserts_df[p_src_modified_field])
    inserts_df = inserts_df.withColumn("ELT_VALID_TO", lit(None).cast(TimestampType()))
    inserts_df = inserts_df.withColumn("Current_Record_Flag", when(col("ELT_VALID_TO").isNull(), 1).otherwise(0))
    return inserts_df

StatementMeta(, 5013aa81-5a5c-46a9-ae5c-70a5977c9eff, 16, Finished, Available, Finished)

## Function to Get Updates

In [15]:
def get_updates(tmp_stg_df, silver_df_join, p_src_modified_field):
    updates_df = tmp_stg_df.join(silver_df_join, tmp_stg_df.ELT_UNIQUE_KEY == silver_df_join.SILVER_DF_ELT_UNIQUE_KEY, "inner")
    updates_df = updates_df.filter(col("SILVER_DF_ELT_VALID_FROM") < col(p_src_modified_field))
    updates_df = updates_df.drop("ELT_SCD_HASH")
    updates_df = updates_df.withColumn("ELT_SCD_HASH", updates_df["SILVER_DF_ELT_SCD_HASH"])
    updates_df = updates_df.select(tmp_stg_df.columns)
    updates_df = updates_df.withColumn("ELT_VALID_FROM", updates_df[p_src_modified_field])
    updates_df = updates_df.withColumn("ELT_VALID_TO", updates_df[p_src_modified_field])
    updates_df = updates_df.withColumn("Current_Record_Flag", lit(0))
    return updates_df

StatementMeta(, 5013aa81-5a5c-46a9-ae5c-70a5977c9eff, 17, Finished, Available, Finished)

## Function to Perform Merge

In [16]:
def perform_merge(merge_df, tgt_table_name):
    silver_delta = DeltaTable.forName(spark, tgt_table_name)
    target_columns = silver_delta.toDF().columns
    updated_dict_insert_clause = {col: f"src.{col}" if col in merge_df.columns else "NULL" for col in target_columns}
    silver_delta.alias('tgt') \
        .merge(
            merge_df.alias('src'),
            'tgt.ELT_SCD_HASH = src.ELT_SCD_HASH'
        ) \
        .whenMatchedUpdate(
            condition="tgt.ELT_VALID_TO IS NULL",
            set={
                "ELT_VALID_TO": "src.ELT_VALID_TO",
                "Current_Record_Flag": "src.Current_Record_Flag"
            }
        ) \
        .whenNotMatchedInsert(
            values=updated_dict_insert_clause
        ) \
        .execute()

StatementMeta(, 5013aa81-5a5c-46a9-ae5c-70a5977c9eff, 18, Finished, Available, Finished)

## Function to Deduplicate and Save Data

In [17]:
def dedup_and_save(merge_df, tgt_table_name):
    dedup_df = merge_df.dropDuplicates()
    dedup_df.write.format("delta").option("mergeSchema", "true").mode("overwrite").saveAsTable(tgt_table_name)

StatementMeta(, 5013aa81-5a5c-46a9-ae5c-70a5977c9eff, 19, Finished, Available, Finished)

## Function to Merge Data to Silver Layer

In [18]:
def merge_to_silver(bronze_df, tgt_table_name, p_src_modified_field):
    silver_df = spark.read.format("delta").table(tgt_table_name)
    silver_df_latest = silver_df.filter(col("ELT_VALID_TO").isNull())
    silver_df_join = silver_df_latest.select([col(col_name).alias(f"SILVER_DF_{col_name}") for col_name in silver_df_latest.columns])
    tmp_stg_df = bronze_df.join(silver_df_join, bronze_df.ELT_UNIQUE_KEY == silver_df_join.SILVER_DF_ELT_UNIQUE_KEY, 'left')
    tmp_stg_df = tmp_stg_df.filter((col(p_src_modified_field) > col("SILVER_DF_ELT_VALID_FROM")) | col("SILVER_DF_ELT_VALID_FROM").isNull())
    tmp_stg_df = tmp_stg_df.select(bronze_df.columns)
    window_spec = Window.partitionBy("ELT_UNIQUE_KEY").orderBy(col(p_src_modified_field).asc())
    tmp_stg_df = tmp_stg_df.withColumn("ELT_ROW_NBR", row_number().over(window_spec)).filter(col("ELT_ROW_NBR") == 1)

    inserts_df = get_inserts(tmp_stg_df, silver_df_join, p_src_modified_field)
    updates_df = get_updates(tmp_stg_df, silver_df_join, p_src_modified_field)
    merge_df = inserts_df.unionAll(updates_df)
    perform_merge(merge_df, tgt_table_name)

StatementMeta(, 5013aa81-5a5c-46a9-ae5c-70a5977c9eff, 20, Finished, Available, Finished)

## Function for Initial Merge to Silver Layer

In [19]:
def initial_merge_to_silver(bronze_df, tgt_table_name, p_src_modified_field):
    window_spec = Window.partitionBy("ELT_UNIQUE_KEY").orderBy(col(p_src_modified_field).asc())
    tmp_stg_df = bronze_df.withColumn("ELT_ROW_NBR", row_number().over(window_spec)).filter(col("ELT_ROW_NBR") == 1)
    merge_df = tmp_stg_df.withColumn("ELT_VALID_FROM", tmp_stg_df[p_src_modified_field])
    merge_df = merge_df.withColumn("ELT_VALID_TO", lit(None).cast(TimestampType()))
    merge_df = merge_df.withColumn("Current_Record_Flag", when(col("ELT_VALID_TO").isNull(), 1).otherwise(0))
    dedup_and_save(merge_df, tgt_table_name)

StatementMeta(, 5013aa81-5a5c-46a9-ae5c-70a5977c9eff, 21, Finished, Available, Finished)

## Function to Prepare Bronze Data for Merge to Silver Layer

In [20]:
def prepare_bronze_data_for_merge(p_src_system_abbr, p_src_db_name, p_src_entity_name, p_tgt_db_name, p_tgt_entity_name, p_merge_key_cols, p_src_modified_field):
    # Set up staging and target parameters
    src_system_code = f"{p_src_system_abbr}"
    stg_table_name = f"{p_src_db_name}.{p_src_entity_name}"
    tgt_table_name = f"{p_tgt_db_name}.{p_tgt_entity_name}"
    concat_columns = p_merge_key_cols.split(',')
    concat_scd_columns = concat_columns + [p_src_modified_field]

    # Check if bronze table exists
    if spark.catalog.tableExists(stg_table_name):
        bronze_df = spark.read.format("delta").table(stg_table_name)
        bronze_df = bronze_df.withColumn("concat_key", concat_ws("|", *[bronze_df[x] for x in concat_columns]))
        bronze_df = bronze_df.withColumn("concat_scd_key", concat_ws("|", *[bronze_df[x] for x in concat_scd_columns]))
        bronze_df = bronze_df.withColumn("ELT_UNIQUE_KEY", sha2(bronze_df.concat_key, 256).cast(StringType()))
        bronze_df = bronze_df.withColumn("ELT_SCD_HASH", sha2(bronze_df.concat_scd_key, 256).cast(StringType()))
        bronze_df = bronze_df.withColumn("SOURCE_SYSTEM_CODE", lit(src_system_code))
        bronze_df = bronze_df.drop("concat_key", "concat_scd_key")

        # Check if silver table exists
        if spark.catalog.tableExists(tgt_table_name):
            merge_to_silver(bronze_df, tgt_table_name, p_src_modified_field)
        else:
            initial_merge_to_silver(bronze_df, tgt_table_name, p_src_modified_field)

StatementMeta(, 5013aa81-5a5c-46a9-ae5c-70a5977c9eff, 22, Finished, Available, Finished)

## Run the Main Function

In [21]:
# Call the function to prepare bronze data for merge to silver layer
prepare_bronze_data_for_merge(p_src_system_abbr, p_src_db_name, p_src_entity_name, p_tgt_db_name, p_tgt_entity_name, p_merge_key_cols, p_src_modified_field)

StatementMeta(, 5013aa81-5a5c-46a9-ae5c-70a5977c9eff, 23, Finished, Available, Finished)

## Collect ELT load time for futher use in the pipeline

In [22]:
from datetime import datetime

# Collect the Max load_datetime from the udpated target table
modified_field_value_df = spark.sql(f"SELECT date_format(max({p_src_modified_field}), 'yyyy-MM-dd HH:mm:ss') FROM {p_tgt_db_name}.{p_tgt_entity_name}")
elt_load_time_str = modified_field_value_df.collect()[0][0]

# Exit the notebook and pass the elt_load_time variable
mssparkutils.notebook.exit({"elt_load_time": elt_load_time_str})

StatementMeta(, 5013aa81-5a5c-46a9-ae5c-70a5977c9eff, 24, Finished, Available, Finished)

ExitValue: {'elt_load_time': '2025-06-24 10:21:58'}